LEVEL 2: Establishing the Base Vectors
   To calculate depth and lateral error, everything must be relative to the
   `socket_mouth`. How do you compute the socket's internal hole, and the
   vector representing the plug's position relative to the socket mouth...

CONCEPTUAL HINTS:
   - Vectors are just directional arrows calculated by subtracing the origin
     point from the destination point.
   - Let M = `socket_mouth`, B = `socket_bottom`, and P = `plug_tip`
   - The socket axis vector is B - M.
   - The relative plug vector is P - M.

SYNTACTICAL HINTS:
   - Declare two local `double` arrays of size 3 (e.g., `double socket_axis[3];`
     and `double plug_rel[3]`;).
   - You can use a `for (int i = 0; i < 3; i++)` loop to subtract the components
     cleanly, or manually unroll `[0]`, `[1]`, `[2]`...


```c
double socket_axis[3];
double plug_rel[3];

for (int i = 0; i < 3; i++) {
    socket_axis[i] = geom->socket_bottom[i] - geom->socket_mouth[i];
    plug_rel[i]    = geom->plug_tip - geom->socket_mouth[i];
}
```

LEVEL 3: Axial Depth (Vector Projection)
   How do you determine exactly how far "down the hole" the plug has traveled
   , assigning it to `score->axial_depth`?

CONCEPTUAL HINTS:
   - You need to project the relative plug vector onto the socket axis.
   - First, normalise the socket axis (scale it so its total length is exactly
     1).
   - Then, take the DOT PRODUCT of the relative plug vector and the normalized
     socket axis. The scalar result is your depth.

SYNTACTICAL HINTS:
   - To normalise a vector, find its length (using `sqrt` of the sum of sqaured
     components), then divide each `socket_axis[i]` by that length.
   - A dot product is the sum of the multiplied corresponding coomponents:
     (x1 * x2) + (y1 * y2) + (z1 * z2)

```c
#include <math.h>

TrailScore *score;
double socket_axis[3];
double plug_rel[3];

for (int i = 0; i < 3; i++) {
    socket_axis[i] = geom->socket_bottom[i] - geom->socket_mouth[i];
    plug_rel[i]    = geom->plug_tip - geom->socket_mouth[i];
}

double norm_axis[3];
double socket_axis_length = sqrt(socket_axis[0]*socket_axis[0] + 
                                 socket_axis[1]*socket_axis[1] + 
                                 socket_axis[2]*socket_axis[2]);

for (int i = 0; i < 3; i++) {
    norm_axis[i] = socket_axis[i] / socket_axis_length;
}

score->axial_depth = (plug_reg[0] * norm_axis[0]) +
                     (plug_reg[1] * norm_axis[1]) +
                     (plug_reg[2] * norm_axis[2]);
```



---

LEVEL 4: Lateral Error (Vector Rejection)
   How do you calculate how far off-center the plug is from the socket axis,
   assigning it to `score->lateral_error`?

CONCEPTUAL HINTS:
   - You know where the plug is (`plug_rel`), and you know how far down the axis
     it has traveled (`axial_depth`)
   - If you take the normalized socket axis, multiply it by the `axial_depth`,
     you get a vector pointing to the "center" of the hole at the plug's current
     depth.
   - Subtract thi center vector from the actual `plug_rel` vector. The result is
     the lateral deviation vector. The length (magnitude) of this deviation
     error is your error.

SYNTACTICAL HINTS:
   - Declare `double lateral_vec[3];`     
   - Inside a loop, the math is: `plug_rel[i] - (score->axial_depth * norm_axis[i]);`
   - Finally, calculate the Euclidean length of `lateral_vec` using `sqrt()`

```c

```

---

   This code is implementing the exact AIC-inspired scoring system you outlined
   in your handoff documents. It cleanly separates the physics (computed 
   geometry) from the grading rubric (scoring).

1. `descending_linear_score` (The Penalty Slope)
   In many benchmarks, "less is more." You want the robot to complete the task
   faster (lower duration), with less movement (lower path length), and less
   shaking (lower jerk).

   This function creates a graded "penalty slope" rather than a harsh pass/fail.
   It behaves in three distinct phases:
   1. THE PERFECT PLATEAU: If the robot's performance `value` is less than or 
      equal to `full_score_value`, it was fast/efficient enough to earn 
      `max_points`. There is no extra reward for being imposssibly fast.
   2. THE VALLEY OF ZERO: If the `value` exceeds the `zero_value_score`, the
      performance was so slow or inefficient that it earns `0.0`.
   3. THE PENALTY SLOPE: If the `value` lands in between, the function 
      calculates exactly where it sits on the slope using linear interpolation:

   ...
   WHY WRITE IT THIS WAY? It prevents a hard cutoff where a robot that takes
   4.9 seconds gets 12 points, but a robot taking 5.1 seconds gets 0 points.
   It smooths the scoring landscape.


2. `eval_score_trial` (The Grading Rubric)
   This function takes the physical state (the geometry) and assigns the actual
   Tier 1, Tier 2, and Tier 3 points. Notice how it structures the priorities:
   - TIER 1 (The Baseline): Automatically grants 1.0 point just for the trail
     running to complletion without segmentation fault.
   - TIER 3 (Task Success): This is the heavy hitter, evaluated before Tier 2.
     It checks a hierarchy of success:
      - Full Insertion: Laterally aligned AND plunged deep enough Awards 75 
        points. 
      - Partial Insertion: Laterally aligned AND partially inside the hole.
        Awards a base 38 points, plus up to 12 extra points depending on the
        fraction of the depth achieved.
      - Proximity: Did not make it into the hole, but gives up to 25 point
        based on how close the plug tip got to the socket.
   - TIER 2 (MOTION QUALITY): The code explicitly wraps this in 
     `if (score->tier3 > 0.0)`. This is a critical design choice! It means 
     you DO NOT AWARD STYLE POINTS for complete failures. If the robot dances
     beautifully but misses the hole entirely, Tier 2 is ignored.
      - If the robot made progress, Tier 2 runs your `descending_linear_score`
        for duration, smoothness (currently mocked to 0.0), and path efficiency.
      - Finally, it subtracts flat penalties for any retries triggered by your 
        staged controller.
        
          



---

   This specific conditon acts as the boundary check to prevent a "partial
   insertion" from being graded as a "full insertion". Think of 
   `socket_depth - config->depth_tol_m` as the finish line: represents the total
   depth of the hole minus a tiny, acceptable margin of error. By checking if
   the plug's current depth (`score->axial_depth`) is strictly less than this 
   finish line, the code is fonriming that the 

   fabs == floating-point absolute value.

   The `zero_score_value` represents the absolute failure threshold for a 
   specific motion metric, marking the exact point where the robot's performance
   becomes too slow, too jerky, or too inefficient to earn any points at all. In
   the context of your `descending_linear_score` function, it anchors the 
   bottom of your "penalty slope"; if the robot's actual performance value meets
   or exceeds this number--like taking 20 seconds on a task where 20 is the 
   absolute cutoff--it receives a receives a flat 0.0 for that specific metric.
   This mechanicsm prevents heavily flaws movements from artificially inflating
   the total score, ensuring that only reasonably efficient run receive
   partial or full credit.